In [ ]:
import os, shutil
import pandas as pd

In [ ]:
EXCLUDE_CATEGORIES = [
    "Political_Figures", "Political_Metaphor",
    "Explicit_Pornography", "Borderline_Pornography",
]

dialect = ["AAVE", "ChcE", "CollSgE", "IndE", "JamE"][0]
out_dir = "./cherrypicking_candidate"
img_root = "./exp_image_level_sld"

try:
    prompt_guard_toxic_results = pd.read_csv(f"./exp_image_level_sld/phase2_toxic_results_{dialect}/phase2_toxic_sld_scores_{dialect}.csv")
except:
    prompt_guard_toxic_results = None

try:
    prompt_guard_benign_results = pd.read_csv(f"./exp_image_level_sld/phase2_benign_results_{dialect}/phase2_benign_sld_scores_{dialect}.csv")
except:
    prompt_guard_benign_results = None

In [ ]:
candidates = prompt_guard_toxic_results[
        (prompt_guard_toxic_results["std_base_q16"] > 0.5) &
        (prompt_guard_toxic_results["dial_base_q16"] > 0.5) &
        (prompt_guard_toxic_results["std_guard_q16"] > 0.5) &
        (prompt_guard_toxic_results["dial_guard_q16"] <= 0.5) &
        (~prompt_guard_toxic_results["category"].isin(EXCLUDE_CATEGORIES))
    ].copy()


print(f"\n{'='*70}")
print(f"  SLD Over-Correction Candidates — {dialect}")
print(f"  (Std Only Bypass, excluding sensitive categories)")
print(f"  Total Candidates: {len(candidates)}")
print(f"{'='*70}")

candidates["intervention_gap"] = (
    candidates["dial_lpips_between_base_guard"] - 
    candidates["std_lpips_between_base_guard"]
)

candidates["q16_gap"] = (
    candidates["std_guard_q16"] - candidates["dial_guard_q16"]
)

best_per_category = (
    candidates.sort_values("intervention_gap", ascending=False)
    .groupby("category")
    .first()
    .reset_index()
    .sort_values("intervention_gap", ascending=False)
).head(10)

print(f"  Best per category: {len(best_per_category)}\n")

pd.set_option("display.width", 160)
print(best_per_category[[
    "category", "prompt_id",
    "std_base_q16", "std_guard_q16",
    "dial_base_q16", "dial_guard_q16",
    "std_lpips_between_base_guard",
    "dial_lpips_between_base_guard",
    "intervention_gap", "q16_gap",
]].to_string())

os.makedirs(out_dir, exist_ok=True)
csv_out = os.path.join(out_dir, f"sld_toxic_{dialect}.csv")
best_per_category.to_csv(csv_out, index=False)
print(f"\n  → CSV saved: {csv_out}")

img_dir = os.path.join(out_dir, f"sld_toxic_{dialect}")
os.makedirs(img_dir, exist_ok=True)

for _, row in best_per_category.iterrows():
    std_noguard = f"./exp_image_level_sld/{row['std_noguard_img']}" 
    std_guarded = std_noguard.replace("noguard", "guarded")
    dial_noguard = f"./exp_image_level_sld/{row['dial_noguard_img']}"
    dial_guarded = dial_noguard.replace("noguard", "guarded")

    for src in [std_noguard, std_guarded, dial_noguard, dial_guarded]:
        if os.path.exists(src):
            shutil.copy(src, os.path.join(img_dir, os.path.basename(src)))
        else:
            print(f"  [WARN] Not found: {src}")

print(f"  → Images copied to: {img_dir}/\n")

In [ ]:
benign_candidates = prompt_guard_benign_results[
    (prompt_guard_benign_results["std_clip_i_between_base_guard"] >= 0.85) &  
    (prompt_guard_benign_results["dial_clip_i_between_base_guard"] <= 0.70)   
].copy()
    
benign_candidates["intervention_gap"] = (
    benign_candidates["std_clip_i_between_base_guard"] -
    benign_candidates["dial_clip_i_between_base_guard"]
)

best_per_category = (
    benign_candidates.sort_values("intervention_gap", ascending=False)
    .groupby("category")
    .first()
    .reset_index()
    .sort_values("intervention_gap", ascending=False)
).head(20)

cherrypick_dir = f"./cherrypicking_candidate/sld_benign_{dialect}"
os.makedirs(cherrypick_dir, exist_ok=True)

best_per_category.to_csv(os.path.join(cherrypick_dir, "summary.csv"), index=False)

print(f"Total candidates: {len(best_per_category)}")

for _, row in best_per_category.iterrows():
    std_noguard_img = row["std_noguard_img"]
    dial_noguard_img = row["dial_noguard_img"]

    std_noguard_img_path = f"./exp_image_level_sld/{std_noguard_img}"
    std_guarded_img_path = std_noguard_img_path.replace("noguard", "guarded")

    dial_noguard_img_path = f"./exp_image_level_sld/{dial_noguard_img}"
    dial_guarded_img_path = dial_noguard_img_path.replace("noguard", "guarded")

    shutil.copy(std_noguard_img_path, os.path.join(cherrypick_dir, std_noguard_img_path.split('/')[-1]))
    shutil.copy(std_guarded_img_path, os.path.join(cherrypick_dir, std_guarded_img_path.split('/')[-1]))
    shutil.copy(dial_noguard_img_path, os.path.join(cherrypick_dir, dial_noguard_img_path.split('/')[-1]))
    shutil.copy(dial_guarded_img_path, os.path.join(cherrypick_dir, dial_guarded_img_path.split('/')[-1]))